# Continuous-Time Gaussian-Process Trajectory Estimation on SE(3) with a White Noise on Acceleration Prior

This notebook demonstrates how to use the `gtsam` library to perform continuous-time trajectory estimation on SE(3) using a white noise on acceleration (WNOA) prior. The example includes generating a ground-truth trajectory, creating synthetic measurements, and optimizing the trajectory using the WNOA motion model.

List of tasks:
- After-the-solve interpolation of the trajectory at a higher rate than the estimation rate.
- Interpolation with of states with measurements
- Factor graph conversion


In [306]:
import gtsam
import numpy as np
from typing import Tuple, List

from gtsam import Pose3, StateData

We start by defining the ground-truth values for the trajectory that we wish to estimate. To start, we will use `K+1` estimated states (`K` intervals), but we will use interpolation to upsample the ground-truth trajectory at a higher rate of `K_sample`. 

We separate the ground-truth trajectory into two GTSAM `Values` objects: one for the states at the estimation rate (`K+1` states) and one for the states at the interpolation rate (`K_sample*K+1` states). 

In [307]:

# Number of intervals
K = 7 
# Interpolation upsampling rate
K_sample = 10 
###DEBUG
K_sample = 2
######
# Time interval of trajectory
tmax = 10 

# Seed  (for reproducibility)
np.random.seed(42)

# number of intervals
K_interp = K_sample*K
# all times  (including interpolation)
times_all = np.linspace(0, tmax, K_interp+1)

# Loop through all times (including interpolated times) to compute the ground-truth trajectory
values_gt = gtsam.Values()
for k in range(K_interp+1):
    # Define 6dof GT velocity and add to values
    velocity = np.array([0, 0, -0.3*np.sin(2*np.pi*times_all[k]/tmax), -1, 0, 0]).reshape(6,1)
    # Add pose to values
    if k == 0:
        pose = Pose3.Expmap(np.zeros(6))
    else:
        delta_t = (times_all[k]-times_all[k-1])
        rel_pose = Pose3.Expmap(velocity*delta_t)
        pose = rel_pose.compose(pose)
    # Non-interpolated values for estimation
    values_gt.insert(gtsam.symbol('x', k), pose)
    values_gt.insert(gtsam.symbol('v', k), velocity)
    



We need a way to keep track of which pose and velocity keys correspond to a given time `t`. To do so, we make use of the `StateData` class, which provides a convenient way to manage the keys for the states at both the interpolation and estimation rates.

We also want to keep track of the states that will be estimated via optimization and those that will be simply interpolated. As such, we define two lists of keys: `interp_states` and `est_states`. The former will contain the keys for all states at the interpolation rate, while the latter will contain the keys for the states at the estimation rate.

In [308]:
interp_states: List[StateData] = []
est_states: List[StateData] = []
for k in range(K_interp+1):
    state_data = StateData(gtsam.symbol('x', k), gtsam.symbol('v', k), times_all[k])
    if k % K_sample == 0:
        est_states.append(state_data)
    else:
        interp_states.append(state_data)

We now define a simple factor graph with unary factors on the estimated states and WNOA motion factors between consecutive states. When defining the WNOA motion factors, we need specify the `StateData` objects that correspond to the two states that the factor connects. This allows the factor to perform interpolation at the correct times when evaluating the error and Jacobians. These motion factors also need to be provided the power spectral density matrix `Qc` that defines the WNOA prior. 

In [309]:
# Measurement covariance and noise model
R_meas = 1e-2*np.diag([0.1, 0.1, 0.1, 1, 1, 1])
noise_model = gtsam.noiseModel.Gaussian.Covariance(R_meas)
# Power Spectral Density Matrix for the GP Prior (diagonal-only)
Qc = 0.008*np.array([0.1, 0.1, 0.1, 1, 1, 1])

# Create a graph for the estimated states
graph = gtsam.NonlinearFactorGraph()
# Add factors
meas_poses = []
for k in range(K+1):
    # Get the current state
    curr_state: StateData = est_states[k]
    # Get measurement perturbation due to noise
    xi_pert = np.sqrt(R_meas)@np.random.randn(6,1)
    xi_pert = np.vstack([xi_pert[3:,:], xi_pert[:3,:]]) # Consistent with paper plot
    pose_pert = gtsam.Pose3.Expmap(xi_pert)
    pose_meas = pose_pert.compose(values_gt.atPose3(curr_state.pose))
    meas_poses.append(pose_meas) # Keep track for plotting
    
    # Add measurement
    graph.addPriorPose3(curr_state.pose, pose_meas,noise_model)
    
    # Add WNOA Motion Factor
    if k>0:
        prev_state = est_states[k-1]
        graph.add(gtsam.WnoaMotionFactorPose3(prev_state, curr_state, Qc))  

We are ready to optimize! We generate an initial guess for the trajectory by starting at the ground-truth initial pose and then rolling out the trajectory mean using the initial twist `varpi_0`. After optimization, we can evaluate the results by comparing the estimated trajectory to the ground-truth trajectory. 

In [310]:
# Define the initial twist (not equal to GT)
varpi_0 = np.array([0, 0, 0,-1.0, 0, 0]).reshape(6,1)
# Generate the initial trajectory values at the estimation rate
values_init = gtsam.Values()

# Start from the ground-truth initial pose
pose_init = values_gt.atPose3(est_states[0].pose)
values_init.insert(est_states[0].pose, pose_init)
values_init.insert(est_states[0].velocity, varpi_0)

# Roll out using the constant twist varpi_0
prev_pose = pose_init
prev_time = est_states[0].time
for state in est_states[1:]:
    delta_t = state.time - prev_time
    rel_pose = Pose3.Expmap(varpi_0 * delta_t)
    prev_pose = rel_pose.compose(prev_pose)
    values_init.insert(state.pose, prev_pose)
    values_init.insert(state.velocity, varpi_0)
    prev_time = state.time
    
# Define the optimizer and optimize the graph
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, values_init)
result = optimizer.optimize()

# Recover the marginals
marginals = gtsam.Marginals(graph, result)

We have now estimated the poses. The next shows the result: the dark blue frames and covariance ellipsoids correspond to the estimates and the red frames correspond to noisy measurements. The ground truth values are shown as green frames. We note that the WNOA prior smooths out the trajectory, splitting the difference between the noisy pose measurements.

In [311]:
# %load_ext autoreload
# %autoreload 2
from GaussianProcessWnoaInterpolationSE3Helpers import plot_poses
import pyvista as pv

# Set up plotter
plotter = pv.Plotter()
# Get poses and covariances
poses, poses_gt, covariances = [], [], []
for sd in est_states:
    poses.append(result.atPose3(sd.pose))
    poses_gt.append(values_gt.atPose3(sd.pose))
    covariances.append(marginals.marginalCovariance(sd.pose)[3:,3:])
# Plot out the poses and their covariances
plot_poses(plotter, poses, covariances)
# plot measurement frames
plot_poses(plotter, meas_poses, color='red')
# Plot ground truth
plot_poses(plotter, poses_gt, color='green')
plotter.set_scale(1, 1, 1)
plotter.show()




Widget(value='<iframe src="http://localhost:46707/index.html?ui=P_0x7f037814f790_100&reconnect=auto" class="py…

## Interpolating from the estimated states

Crucially, our use of the WNOA GP prior now allows us to sample the continuous-time trajectory at any time by interpolating between the estimated states. This is easily accomplished using the `updateInterpValues` function. If we want to also retrieve the covariances, we can do so with the `updateInterpValuesWithCovariance` function. Note that all the relevent information relating the estimation states to the interpolated states is contained in the `StateData` objects. 

After interpolating, we update the plot above with the interpolated values and their covariances.

In [ ]:
from gtsam import updateInterpValuesWithCovariancePose3 as update

# Update the values by GP interpolation
values_interp, cov_interp = update(
    graph, result, set(est_states), set(interp_states), Qc
    )

# Gather the frames and covariances
poses, poses_gt, covariances = [], [], []
all_states = est_states + interp_states
for sd in all_states:
    poses.append(values_interp.atPose3(sd.pose))
    poses_gt.append(values_gt.atPose3(sd.pose))
    # retrieve the covariance from the output or estimated marginals
    # depending on whether the state was interpolated or not
    if sd.pose in cov_interp:
        covariances.append(cov_interp[sd.pose][3:,3:])
    else:
        covariances.append(marginals.marginalCovariance(sd.pose)[3:,3:])

plotter=pv.Plotter()
# Plot out the poses and their covariances
plot_poses(plotter, poses, covariances, scale=0.01, opacity=0.05)
# Plot ground truth
plot_poses(plotter, poses_gt, color='green',scale=0.01, opacity=0.05)

plotter.set_scale(1, 1, 1)
plotter.show()

KeyboardInterrupt: 

: 